# Extensions de Nexus : NER et RAG

Ce notebook présente deux extensions du modèle Nexus de base :

- **NER (Named Entity Recognition)**  identifier automatiquement les entités biomédicales (médicaments, maladies, gènes) dans un texte, avec le modèle `biomedbert_ner_bc5cdr`.
- **RAG (Retrieval-Augmented Generation)** combiner le modèle de langue Nexus avec une recherche documentaire pour répondre à des questions précises en s'appuyant sur un corpus PubMed.

Ces deux briques transforment Nexus d'un simple modèle de langue en un véritable **assistant documentaire spécialisé** pour le secteur pharmaceutique.

In [ ]:
!pip install -q transformers torch scikit-learn requests

## 1. NER biomédical BioBERT fine-tuné sur BC5CDR

On charge `Glasgow-AI4BioMed/bioner_bc5cdr` : un BERT pré-entraîné sur PubMed puis fine-tuné sur **BC5CDR** (BioCreative V), qui annote chimiques (médicaments) et maladies.

L'option `aggregation_strategy="simple"` regroupe les sous-tokens BERT en entités complètes lisibles.

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="Glasgow-AI4BioMed/bioner_bc5cdr",
    aggregation_strategy="simple"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Test sur un cas réel

On donne au modèle deux phrases typiques d'un compte-rendu médical. Le pipeline doit identifier `aspirin`, `metformin` comme **CHEMICAL** et `myocardial infarction`, `type 2 diabetes` comme **DISEASE**.

In [ ]:
texte = """
The patient received aspirin after a diagnosis of myocardial infarction.
Treatment with metformin reduced glucose levels in patients with type 2 diabetes.
"""

resultats = ner(texte)

for entite in resultats:
    print(entite["word"], "->", entite["entity_group"], "score:", round(entite["score"], 3))


aspirin -> Chemical score: 0.999
myocardial infarction -> Disease score: 0.999
metformin -> Chemical score: 0.999
glucose -> Chemical score: 0.999
type 2 diabetes -> Disease score: 0.997


## 2. RAG  Récupération + génération augmentée

Le modèle Nexus seul peut halluciner. Le **RAG** corrige ça en deux temps :
1. **Retrieval** — étant donnée une question, on récupère les passages les plus pertinents dans un corpus.
2. **Generation** — le modèle génère sa réponse en s'appuyant uniquement sur ces passages.

### Corpus de démonstration

Trois mini-documents biomédicaux servent de base de connaissances pour la suite.

In [ ]:
documents = [
    "Aspirin is used to reduce pain, fever, and inflammation. It is also used after myocardial infarction.",
    "Metformin is commonly used for type 2 diabetes and helps reduce blood glucose levels.",
    "Ibuprofen is a nonsteroidal anti-inflammatory drug used for pain and fever.",
    "The estimated cost of the analysis depends on the quantity, machines used, and number of technicians."
]


### Retriever similarité TF-IDF

On vectorise chaque document avec TF-IDF puis on calcule la **similarité cosinus** entre la question et chaque document. Les `top_k` documents les plus proches sont renvoyés.

En production, TF-IDF serait remplacé par des embeddings denses (Sentence-BERT, BioBERT) pour gérer la synonymie sémantique.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
X_docs = vectorizer.fit_transform(documents)

def rechercher_passages(question, documents, top_k=2):
    q_vec = vectorizer.transform([question])
    scores = cosine_similarity(q_vec, X_docs)[0]
    indices = scores.argsort()[::-1][:top_k]

    resultats = []
    for i in indices:
        resultats.append((documents[i], scores[i]))

    return resultats


### Première requête : retrouver le bon document

On pose une question sur le diabète. Le retriever doit ranker en tête le document parlant de `metformin` et `type 2 diabetes`.

In [ ]:
question = "Which treatment is used for diabetes?"

passages = rechercher_passages(question, documents)

for passage, score in passages:
    print("Score :", round(score, 3))
    print(passage)
    print()

Score : 0.475
Metformin is commonly used for type 2 diabetes and helps reduce blood glucose levels.

Score : 0.307
Ibuprofen is a nonsteroidal anti-inflammatory drug used for pain and fever.



### Construction du contexte à fournir au générateur

On concatène les `top_k` passages retrouvés en un seul **contexte**. Ce contexte est ensuite ce qu'on injecterait en prompt à Nexus pour qu'il génère une réponse ancrée dans le corpus, et non hallucinée.

In [ ]:
question = "Which drug is used for diabetes?"

passages = rechercher_passages(question, documents, top_k=2)
contexte = " ".join([p[0] for p in passages])

print("Passages retrouvés :")
print(contexte)

print("\nEntités biomédicales détectées :")
entites = ner(contexte)

for e in entites:
    print(e["word"], "->", e["entity_group"], round(e["score"], 3))

Passages retrouvés :
Ibuprofen is a nonsteroidal anti-inflammatory drug used for pain and fever. Metformin is commonly used for type 2 diabetes and helps reduce blood glucose levels.

Entités biomédicales détectées :
ibuprofen -> Chemical 0.998
pain -> Disease 1.0
fever -> Disease 1.0
metformin -> Chemical 0.998
type 2 diabetes -> Disease 0.999
glucose -> Chemical 0.93


## Conclusion

- Le **NER** structure les textes médicaux bruts → extraction de médicaments et pathologies.
- Le **RAG** ancre la génération dans un corpus réel → réponses vérifiables.

Ensemble, ils répondent aux deux limites d'un LM seul en contexte pharmaceutique : **flou sémantique** et **hallucinations**. La prochaine étape serait de brancher le retriever sur un index PubMed complet et de remplacer TF-IDF par des embeddings denses.